In [20]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [36]:
X_Tr = pd.read_csv('train.csv')
X_Te = pd.read_csv('test.csv')

In [ ]:
#PRE_PROCESSING

In [37]:
X_Tr.fillna(X_Tr.mean(),inplace = True)
X_Tr.isnull().sum()
X_Te.fillna(X_Te.mean(),inplace = True)
X_Te.isnull().sum()

C:\Users\MUHAMM~1\AppData\Local\Temp/ipykernel_13568/705902647.py:1: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  X_Tr.fillna(X_Tr.mean(),inplace = True)
C:\Users\MUHAMM~1\AppData\Local\Temp/ipykernel_13568/705902647.py:3: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  X_Te.fillna(X_Te.mean(),inplace = True)


MSSubClass       0
MSZoning         4
LotFrontage      0
LotArea          0
Street           0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         1
SaleCondition    0
Length: 79, dtype: int64

In [38]:
LabelEnc = LabelEncoder()
for column in X_Tr.select_dtypes(include = 'object').columns:
    X_Tr[column] = LabelEnc.fit_transform(X_Tr[column])
    
for column in X_Te.select_dtypes(include = 'object').columns:
    X_Te[column] = LabelEnc.fit_transform(X_Te[column])

In [39]:
X_Tr.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,3,65.0,8450,1,2,3,3,0,4,...,0,3,4,4,0,2,2008,8,4,208500
1,20,3,80.0,9600,1,2,3,3,0,2,...,0,3,4,4,0,5,2007,8,4,181500
2,60,3,68.0,11250,1,2,0,3,0,4,...,0,3,4,4,0,9,2008,8,4,223500
3,70,3,60.0,9550,1,2,0,3,0,0,...,0,3,4,4,0,2,2006,8,0,140000
4,60,3,84.0,14260,1,2,0,3,0,2,...,0,3,4,4,0,12,2008,8,4,250000


In [40]:
X_Te.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,20,2,80.0,11622,1,2,3,3,0,4,...,120,0,2,2,3,0,6,2010,8,4
1,20,3,81.0,14267,1,2,0,3,0,0,...,0,0,2,4,0,12500,6,2010,8,4
2,60,3,74.0,13830,1,2,0,3,0,4,...,0,0,2,2,3,0,3,2010,8,4
3,60,3,78.0,9978,1,2,0,3,0,4,...,0,0,2,4,3,0,6,2010,8,4
4,120,3,43.0,5005,1,2,0,1,0,4,...,144,0,2,4,3,0,1,2010,8,4


In [81]:
X_Train = X_Tr.drop('SalePrice', axis = 1)
Y_Train = X_Tr.SalePrice

In [82]:
X_Train = X_Train.to_numpy()
Y_Train = Y_Train.to_numpy()
X_Test = X_Te.to_numpy()

In [83]:
Y_Train = Y_Train / 1000
X_Train = X_Train / 1000
X_Test = X_Test / 1000

In [84]:
print(X_Train.shape, Y_Train.shape, X_Test.shape)
print(X_Train, Y_Train, X_Test)

(1460, 79) (1460,) (1459, 79)
[[0.06  0.003 0.065 ... 2.008 0.008 0.004]
 [0.02  0.003 0.08  ... 2.007 0.008 0.004]
 [0.06  0.003 0.068 ... 2.008 0.008 0.004]
 ...
 [0.07  0.003 0.066 ... 2.01  0.008 0.004]
 [0.02  0.003 0.068 ... 2.01  0.008 0.004]
 [0.02  0.003 0.075 ... 2.008 0.008 0.004]] [208.5   181.5   223.5   ... 266.5   142.125 147.5  ] [[2.000e-02 2.000e-03 8.000e-02 ... 2.010e+00 8.000e-03 4.000e-03]
 [2.000e-02 3.000e-03 8.100e-02 ... 2.010e+00 8.000e-03 4.000e-03]
 [6.000e-02 3.000e-03 7.400e-02 ... 2.010e+00 8.000e-03 4.000e-03]
 ...
 [2.000e-02 3.000e-03 1.600e-01 ... 2.006e+00 8.000e-03 0.000e+00]
 [8.500e-02 3.000e-03 6.200e-02 ... 2.006e+00 8.000e-03 4.000e-03]
 [6.000e-02 3.000e-03 7.400e-02 ... 2.006e+00 8.000e-03 4.000e-03]]


In [85]:
def costFunc(W, b, X, Y, lambda_):
    m, n = X.shape
    cost = 0
    regu = 0
    J_pred = []
    for i in range(len(W)):
        regu += (W[i]) ** 2
    regu = (lambda_ / (2 * m)) * regu
    for i in range(m):
        f_x = np.dot(W, X[i]) + b
        cost += (f_x - Y[i]) ** 2
    cost = ((1 / (2 * m)) * cost) + regu
    return cost

In [86]:
W_in = np.random.randn(X_Train.shape[1], ) #np.array([-1, -2, 1, 1.5, 0.5, -2, 1, -1, 2, -2, 1, 1])

In [87]:
print(W_in)

[ 1.70798215 -0.3953666   0.96656664  1.21266313  1.69070149  0.52882302
  0.6459753   0.00631522 -1.32827494 -0.20674457  0.35332193  0.90356292
  0.52883637  1.29581284 -1.86742976 -0.22565763  0.28089978  0.08999366
 -0.7458004  -1.43245558  0.10876834  0.51400101  0.70435019  0.38523874
  0.38747925 -0.20466633 -0.20830649 -0.03386654 -0.09695089 -0.43872956
  0.72927339 -0.44975867 -1.68686903 -0.11252674  0.15410379  0.56637206
  0.1568371   0.54705956 -0.30489291  0.85748953 -0.42631376  0.13531941
  1.90149716 -0.13452214 -0.63522836 -1.54935621 -0.78187469  0.12507124
 -1.84129474  0.97017683 -1.36805476 -0.31978589  0.35020471  1.23801192
 -0.99940791  1.06915799  0.55968932  0.87467719  1.59975216  0.58853651
 -0.06793325  1.25269611 -0.90745617  0.70475952 -0.74112171 -0.03667467
  2.49299608 -0.26980689  1.76368048  0.49801617 -0.45908102 -1.2668422
 -0.83670929 -0.50756804  0.62147312 -2.46149567 -0.62101367  0.46317999
  1.82201961]


In [62]:
def calcGradients(W, b, X, Y, lambda_):
    m, n = X.shape
    dJ_dw = 0
    dJ_db = 0
    regu = 0
    for i in range(len(W)):
        regu += (W[i])
    regu = (lambda_ / (m)) * regu
    for i in range(m):
        f_x = np.dot(W, X[i]) + b
        dJ_dw += (f_x - Y[i]) * X[i]
        dJ_db += (f_x - Y[i])
    dJ_dw = ((1 / m) * dJ_dw) + regu
    dJ_db = (1 / m) * dJ_db
    return dJ_dw, dJ_db

In [63]:
def gradientDescent(iterations, W, b, X, Y, lambda_, alpha):
    for i in range(iterations):
        dJ_dw, dJ_db = calcGradients(W, b, X, Y, lambda_)
        W = W - alpha * dJ_dw
        b = b - alpha * dJ_db
        if (i % 50 == 0):
            J = costFunc(W, b, X, Y, lambda_)
            print("The cost at", i, "iteration is", J)
    return W, b

In [93]:
def predict(W, b, X):
    m, n = X.shape
    pred = []
    for i in range(m):
        f_x = np.dot(W, X[i]) + b
        pred.append(f_x)
    return pred

In [110]:
W, b = gradientDescent(215000, W_in, 0.548, X_Train, Y_Train, 0.03, 0.008)
print("Optimum parameters are", W, b)

The cost at 0 iteration is 12840.059440269628
The cost at 50 iteration is 1954.4686210186753
The cost at 100 iteration is 1732.199163389142
The cost at 150 iteration is 1568.8277009808712
The cost at 200 iteration is 1447.6220996844993
The cost at 250 iteration is 1357.4763461192385
The cost at 300 iteration is 1290.2187513136892
The cost at 350 iteration is 1239.8355061316827
The cost at 400 iteration is 1201.9001573823518
The cost at 450 iteration is 1173.1540424094878
The cost at 500 iteration is 1151.1977153694925
The cost at 550 iteration is 1134.2639851836705
The cost at 600 iteration is 1121.0509659027603
The cost at 650 iteration is 1110.5992596138465
The cost at 700 iteration is 1102.201596337718
The cost at 750 iteration is 1095.336346077696
The cost at 800 iteration is 1089.6185903493672
The cost at 850 iteration is 1084.7641110062461
The cost at 900 iteration is 1080.56288236999
The cost at 950 iteration is 1076.859555729588
The cost at 1000 iteration is 1073.5390893045183


The cost at 8550 iteration is 969.7912973162131
The cost at 8600 iteration is 969.6519127808258
The cost at 8650 iteration is 969.5139132707819
The cost at 8700 iteration is 969.3772695997249
The cost at 8750 iteration is 969.2419532850116
The cost at 8800 iteration is 969.1079365299315
The cost at 8850 iteration is 968.9751922063741
The cost at 8900 iteration is 968.8436938379566
The cost at 8950 iteration is 968.7134155835719
The cost at 9000 iteration is 968.5843322213888
The cost at 9050 iteration is 968.4564191332422
The cost at 9100 iteration is 968.329652289449
The cost at 9150 iteration is 968.2040082340078
The cost at 9200 iteration is 968.0794640701804
The cost at 9250 iteration is 967.9559974464566
The cost at 9300 iteration is 967.8335865428534
The cost at 9350 iteration is 967.7122100576155
The cost at 9400 iteration is 967.59184719422
The cost at 9450 iteration is 967.4724776487249
The cost at 9500 iteration is 967.3540815974555
The cost at 9550 iteration is 967.236639685

The cost at 17000 iteration is 954.9488771866695
The cost at 17050 iteration is 954.883867184425
The cost at 17100 iteration is 954.8189832449049
The cost at 17150 iteration is 954.7542244324562
The cost at 17200 iteration is 954.6895898250091
The cost at 17250 iteration is 954.6250785138166
The cost at 17300 iteration is 954.5606896031836
The cost at 17350 iteration is 954.496422210211
The cost at 17400 iteration is 954.4322754645547
The cost at 17450 iteration is 954.368248508173
The cost at 17500 iteration is 954.3043404950848
The cost at 17550 iteration is 954.2405505911477
The cost at 17600 iteration is 954.1768779738202
The cost at 17650 iteration is 954.1133218319361
The cost at 17700 iteration is 954.0498813654937
The cost at 17750 iteration is 953.9865557854324
The cost at 17800 iteration is 953.9233443134416
The cost at 17850 iteration is 953.8602461817295
The cost at 17900 iteration is 953.797260632853
The cost at 17950 iteration is 953.734386919495
The cost at 18000 iterati

The cost at 25400 iteration is 945.3321316556908
The cost at 25450 iteration is 945.2808993221842
The cost at 25500 iteration is 945.2297230878014
The cost at 25550 iteration is 945.1786027652327
The cost at 25600 iteration is 945.1275381683045
The cost at 25650 iteration is 945.0765291119895
The cost at 25700 iteration is 945.0255754123644
The cost at 25750 iteration is 944.9746768866307
The cost at 25800 iteration is 944.9238333530772
The cost at 25850 iteration is 944.8730446310859
The cost at 25900 iteration is 944.8223105411116
The cost at 25950 iteration is 944.7716309046817
The cost at 26000 iteration is 944.7210055443774
The cost at 26050 iteration is 944.670434283822
The cost at 26100 iteration is 944.6199169476794
The cost at 26150 iteration is 944.5694533616337
The cost at 26200 iteration is 944.5190433523959
The cost at 26250 iteration is 944.4686867476694
The cost at 26300 iteration is 944.4183833761635
The cost at 26350 iteration is 944.3681330675735
The cost at 26400 ite

The cost at 33800 iteration is 937.3945103574354
The cost at 33850 iteration is 937.3507101326334
The cost at 33900 iteration is 937.3069449413931
The cost at 33950 iteration is 937.263214699267
The cost at 34000 iteration is 937.2195193221269
The cost at 34050 iteration is 937.1758587261738
The cost at 34100 iteration is 937.132232827917
The cost at 34150 iteration is 937.0886415441782
The cost at 34200 iteration is 937.045084792104
The cost at 34250 iteration is 937.0015624891422
The cost at 34300 iteration is 936.9580745530535
The cost at 34350 iteration is 936.9146209019083
The cost at 34400 iteration is 936.8712014540807
The cost at 34450 iteration is 936.8278161282511
The cost at 34500 iteration is 936.7844648434032
The cost at 34550 iteration is 936.7411475188204
The cost at 34600 iteration is 936.6978640740872
The cost at 34650 iteration is 936.6546144290844
The cost at 34700 iteration is 936.6113985039897
The cost at 34750 iteration is 936.5682162192755
The cost at 34800 itera

...

The cost at 175250 iteration is 863.6571248097982
The cost at 175300 iteration is 863.6396069287729
The cost at 175350 iteration is 863.6220929395848
The cost at 175400 iteration is 863.6045828411649
The cost at 175450 iteration is 863.587076632453
The cost at 175500 iteration is 863.5695743123784
The cost at 175550 iteration is 863.5520758798841
The cost at 175600 iteration is 863.5345813338997
The cost at 175650 iteration is 863.517090673369
The cost at 175700 iteration is 863.4996038972233
The cost at 175750 iteration is 863.4821210044043
The cost at 175800 iteration is 863.464641993849
The cost at 175850 iteration is 863.4471668644953
The cost at 175900 iteration is 863.4296956152855
The cost at 175950 iteration is 863.4122282451558
The cost at 176000 iteration is 863.3947647530493
The cost at 176050 iteration is 863.377305137905
The cost at 176100 iteration is 863.3598493986651
The cost at 176150 iteration is 863.342397534272
The cost at 176200 iteration is 863.3249495436677
The c

The cost at 183500 iteration is 860.8185621113317
The cost at 183550 iteration is 860.8016723205982
The cost at 183600 iteration is 860.7847862523246
The cost at 183650 iteration is 860.7679039055242
The cost at 183700 iteration is 860.7510252792102
The cost at 183750 iteration is 860.7341503723909
The cost at 183800 iteration is 860.7172791840796
The cost at 183850 iteration is 860.7004117132938
The cost at 183900 iteration is 860.6835479590394
The cost at 183950 iteration is 860.6666879203339
The cost at 184000 iteration is 860.6498315961915
The cost at 184050 iteration is 860.6329789856243
The cost at 184100 iteration is 860.6161300876513
The cost at 184150 iteration is 860.5992849012822
The cost at 184200 iteration is 860.5824434255394
The cost at 184250 iteration is 860.5656056594348
The cost at 184300 iteration is 860.5487716019859
The cost at 184350 iteration is 860.5319412522102
The cost at 184400 iteration is 860.5151146091249
The cost at 184450 iteration is 860.4982916717485


The cost at 191750 iteration is 858.0813970058534
The cost at 191800 iteration is 858.0651083462586
The cost at 191850 iteration is 858.0488232514981
The cost at 191900 iteration is 858.0325417206557
The cost at 191950 iteration is 858.0162637528027
The cost at 192000 iteration is 857.9999893470238
The cost at 192050 iteration is 857.9837185023912
The cost at 192100 iteration is 857.9674512179864
The cost at 192150 iteration is 857.9511874928885
The cost at 192200 iteration is 857.9349273261753
The cost at 192250 iteration is 857.9186707169291
The cost at 192300 iteration is 857.9024176642268
The cost at 192350 iteration is 857.8861681671523
The cost at 192400 iteration is 857.8699222247841
The cost at 192450 iteration is 857.8536798362037
The cost at 192500 iteration is 857.8374410004942
The cost at 192550 iteration is 857.8212057167367
The cost at 192600 iteration is 857.8049739840112
The cost at 192650 iteration is 857.7887458014054
The cost at 192700 iteration is 857.7725211679988


The cost at 200000 iteration is 855.4413328346939
The cost at 200050 iteration is 855.4256201525747
The cost at 200100 iteration is 855.4099108878518
The cost at 200150 iteration is 855.3942050396643
The cost at 200200 iteration is 855.3785026071407
The cost at 200250 iteration is 855.3628035894232
The cost at 200300 iteration is 855.3471079856456
The cost at 200350 iteration is 855.3314157949444
The cost at 200400 iteration is 855.315727016455
The cost at 200450 iteration is 855.3000416493124
The cost at 200500 iteration is 855.2843596926565
The cost at 200550 iteration is 855.2686811456235
The cost at 200600 iteration is 855.2530060073531
The cost at 200650 iteration is 855.2373342769807
The cost at 200700 iteration is 855.2216659536459
The cost at 200750 iteration is 855.2060010364884
The cost at 200800 iteration is 855.190339524643
The cost at 200850 iteration is 855.1746814172539
The cost at 200900 iteration is 855.1590267134592
The cost at 200950 iteration is 855.143375412398
The

The cost at 208250 iteration is 852.894351148371
The cost at 208300 iteration is 852.879190863979
The cost at 208350 iteration is 852.8640338584779
The cost at 208400 iteration is 852.8488801310518
The cost at 208450 iteration is 852.8337296808878
The cost at 208500 iteration is 852.8185825071708
The cost at 208550 iteration is 852.8034386090861
The cost at 208600 iteration is 852.7882979858201
The cost at 208650 iteration is 852.7731606365645
The cost at 208700 iteration is 852.758026560502
The cost at 208750 iteration is 852.7428957568233
The cost at 208800 iteration is 852.7277682247147
The cost at 208850 iteration is 852.7126439633633
The cost at 208900 iteration is 852.6975229719609
The cost at 208950 iteration is 852.6824052496956
The cost at 209000 iteration is 852.6672907957525
The cost at 209050 iteration is 852.652179609323
The cost at 209100 iteration is 852.637071689602
The cost at 209150 iteration is 852.6219670357724
The cost at 209200 iteration is 852.6068656470239
The c

In [111]:
y_pred_train = predict(W, b, X_Train)

In [112]:
for i in range(50):
    print(y_pred_train[i], ",", Y_Train[i])

208.496540707996 , 208.5
183.42602625240457 , 181.5
214.3039354746792 , 223.5
165.97153730252896 , 140.0
284.5060302973636 , 250.0
178.71659074031717 , 143.0
260.314645390358 , 307.0
238.70373385305552 , 200.0
164.5281906532368 , 129.9
98.55988635691753 , 118.0
138.36720914124825 , 129.5
286.696305811916 , 345.0
137.94312896583526 , 144.0
237.10149968219687 , 279.5
163.97488215452367 , 157.0
123.20218280658932 , 132.0
145.77906254955136 , 149.0
109.67628296844819 , 90.0
174.49332598263854 , 159.0
141.50392118862467 , 139.0
297.3921650961945 , 325.3
91.45751102616046 , 139.4
246.06012249718012 , 230.0
153.02665991589194 , 129.9
145.19292678538582 , 154.0
256.8286597935306 , 256.3
140.42159630052524 , 134.8
256.50297573791227 , 306.0
215.14605744798362 , 207.5
53.77423482588111 , 68.5
104.73949244845059 , 40.0
143.3976675972693 , 149.35
167.71163373432145 , 179.9
194.32175532681677 , 165.5
228.58487841894785 , 277.5
268.329556801948 , 309.0
180.75045256842512 , 145.0
201.70219086138607 ,

In [113]:
y_pred_test = predict(W, b, X_Test)

In [114]:
for i in range(50):
    print(y_pred_test[i])

152.4065885766066
168.5541030023962
199.91154702497377
202.01732010224956
170.89828475268516
180.7856016330641
188.18424860839602
159.6175823773403
186.93614185733676
143.2436739219281
193.50358624578766
122.96820127196767
124.6896999643665
156.74638774916812
95.33836397796588
283.50536044309763
234.94623773107142
265.6087692312916
250.10131255570977
420.27830332772805
292.4289418425881
184.31211114250286
173.4081823150689
152.7043671571993
148.45640661554975
190.45646497518837
315.9224820040789
243.57902953904122
183.10947467731577
225.61805484084366
195.18588721813657
108.77818022739027
190.11183849789435
277.06819304072474
276.6659238142498
208.29245002606578
182.04935214442978
147.0822941149651
143.03537981776665
155.2380793712404
208.26860790915973
125.82247659291207
254.9487958812921
228.57556452225404
225.00144803929547
225.20578574404843
282.98894191790635
223.16990778442712
165.30714935623624
175.01926716123612
